# 注意力权重可视化

本 Notebook 演示如何可视化 Transformer 模型的注意力权重，帮助理解模型在处理文本时「关注了哪里」。

## 前置要求
- 已完成模型训练（或使用未训练的模型观察随机初始化的注意力权重）
- 已安装 matplotlib

In [ ]:
import sys
import os
sys.path.insert(0, os.path.dirname(os.getcwd()))
os.chdir(os.path.dirname(os.getcwd()))

import torch
import matplotlib.pyplot as plt
import numpy as np

from src.transformer import Transformer
from src.dataset import CharTokenizer, PAD_ID

print('导入完成！')

## 1. 创建模型和分词器

In [ ]:
# 尝试加载训练好的模型，如果没有则使用随机初始化的模型
model_path = 'experiments/baseline/model_best.pt'
device = torch.device('cpu')

if os.path.exists(model_path):
    print(f'加载训练好的模型: {model_path}')
    checkpoint = torch.load(model_path, map_location=device, weights_only=False)
    config = checkpoint['config']
    
    tokenizer = CharTokenizer()
    tokenizer.load('data/vocab.json')
    
    model = Transformer(
        vocab_size=tokenizer.vocab_size,
        d_model=config['model']['d_model'],
        num_heads=config['model']['num_heads'],
        d_ff=config['model']['d_ff'],
        num_layers=config['model']['num_layers'],
    )
    model.load_state_dict(checkpoint['model_state_dict'])
    print('模型加载成功！')
else:
    print('未找到训练模型，使用随机初始化')
    tokenizer = CharTokenizer()
    tokenizer.build_vocab('abcdefghijklmnopqrstuvwxyz ABCDEFGHIJKLMNOPQRSTUVWXYZ.,!?:\n')
    model = Transformer(vocab_size=tokenizer.vocab_size, d_model=64, num_heads=4)

model.eval()
print(f'词表大小: {tokenizer.vocab_size}')

## 2. 获取注意力权重

In [ ]:
# 输入文本
src_text = "ROMEO: O, she doth teach"
tgt_text = " the torches to burn b"

# 编码
src_ids = tokenizer.encode(src_text)
tgt_ids = tokenizer.encode(tgt_text)

src = torch.tensor([src_ids], dtype=torch.long)
tgt = torch.tensor([tgt_ids], dtype=torch.long)

print(f'源序列: {repr(src_text)}')
print(f'目标序列: {repr(tgt_text)}')
print(f'源序列长度: {len(src_ids)}')
print(f'目标序列长度: {len(tgt_ids)}')

# 前向传播获取注意力权重
with torch.no_grad():
    logits, cross_attn_weights = model(src, tgt)

print(f'\n交叉注意力权重形状: {cross_attn_weights.shape}')
print(f'(batch_size, num_heads, tgt_len, src_len)')

## 3. 可视化交叉注意力权重

交叉注意力权重展示了解码器在生成每个目标字符时，关注了源序列中的哪些字符。

In [ ]:
# 取第一个样本的注意力权重
# attn: [num_heads, tgt_len, src_len]
attn = cross_attn_weights[0].numpy()
num_heads = attn.shape[0]

# 创建字符标签
src_chars = list(src_text)
tgt_chars = list(tgt_text)

# 绘制每个 head 的注意力热图
fig, axes = plt.subplots(1, num_heads, figsize=(5 * num_heads, 6))
if num_heads == 1:
    axes = [axes]

for head_idx in range(num_heads):
    ax = axes[head_idx]
    im = ax.imshow(attn[head_idx], cmap='hot', aspect='auto')
    
    # 设置标签
    ax.set_xticks(range(len(src_chars)))
    ax.set_xticklabels(src_chars, fontsize=8, rotation=90)
    ax.set_yticks(range(len(tgt_chars)))
    ax.set_yticklabels(tgt_chars, fontsize=8)
    
    ax.set_title(f'Head {head_idx + 1}', fontsize=12)
    ax.set_xlabel('源序列 (Source)', fontsize=10)
    if head_idx == 0:
        ax.set_ylabel('目标序列 (Target)', fontsize=10)

plt.suptitle('交叉注意力权重热图（Cross-Attention Weights）', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('notebooks/cross_attention_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('图片已保存到 notebooks/cross_attention_heatmap.png')

## 4. 平均注意力权重

将所有 head 的注意力权重取平均，可以看到总体的关注模式。

In [ ]:
# 平均所有 head
avg_attn = attn.mean(axis=0)  # [tgt_len, src_len]

fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(avg_attn, cmap='hot', aspect='auto')

ax.set_xticks(range(len(src_chars)))
ax.set_xticklabels(src_chars, fontsize=10, rotation=90)
ax.set_yticks(range(len(tgt_chars)))
ax.set_yticklabels(tgt_chars, fontsize=10)

ax.set_title('平均交叉注意力权重', fontsize=14)
ax.set_xlabel('源序列 (Source)', fontsize=12)
ax.set_ylabel('目标序列 (Target)', fontsize=12)

plt.colorbar(im, ax=ax, label='注意力权重')
plt.tight_layout()
plt.savefig('notebooks/avg_attention_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('图片已保存到 notebooks/avg_attention_heatmap.png')

## 5. 解读注意力权重

### 如何看这些热图？

- **纵轴**：目标序列中的每个字符（模型正在生成的字符）
- **横轴**：源序列中的每个字符（模型可以参考的内容）
- **颜色深浅**：注意力权重的大小（越亮 = 越关注）

### 期望看到什么？

- **训练良好的模型**：会显示有意义的注意力模式，比如对角线模式（相邻位置的对应关系）
- **随机初始化的模型**：注意力权重接近均匀分布（没有明显的模式）
- **不同的 head**：不同的 head 可能关注不同的模式

### 思考题

1. 不同 head 的注意力模式有什么不同？
2. 模型在生成标点符号时，注意力分布和生成字母时有区别吗？
3. 如果去掉位置编码，注意力模式会怎样变化？